# Multi-horizon dynamics v1
Private, frozen, CPU-only scientific execution. Internet must remain off.


In [ ]:
from pathlib import Path
import os
import runpy

root = Path('/kaggle/working/multi_horizon_frozen_source')
script = root / 'scripts/run_multi_horizon_dynamics_v1.py'
protocol = root / 'research/multi_horizon_dynamics_v1/PROTOCOL.md'
script.parent.mkdir(parents=True, exist_ok=True)
protocol.parent.mkdir(parents=True, exist_ok=True)
script.write_text('"""One-shot frozen multi-horizon synthetic scientific campaign."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport platform\nimport random\nimport sys\nimport time\nfrom pathlib import Path\nfrom typing import Any\n\nDATASET_SEEDS = (20267100, 20267200, 20267300)\nOPTIMIZER_SEEDS = (20267601, 20267602)\nPROTECTED_OFFSETS = {"clean": 11, "corrupted": 12, "shifted": 13}\nTRAIN_EXAMPLES = 512\nDEVELOPMENT_EXAMPLES = 128\nPROTECTED_EXAMPLES = 256\nHISTORY = 6\nHORIZON = 8\nSTATE_DIM = 4\nACTION_DIM = 2\nHIDDEN_DIM = 32\nUPDATES = 200\nBATCH_SIZE = 64\nLEARNING_RATE = 0.002\nMODEL_NAMES = ("direct", "residual", "uncertainty_residual")\nBOOTSTRAP_SAMPLES = 10_000\nBOOTSTRAP_SEED = 20267999\nMAX_WALL_SECONDS = 3_600\nWORST_REGIME_H8_RATIO_MAX = 0.90\nPAIRED_DELTA_CI_UPPER_MAX = 0.0\nCLEAN_H8_REGRESSION_MAX = 0.02\nCOVERAGE90_ERROR_MAX = 0.10\n\nPROTOCOL = {\n    "id": "multi-horizon-dynamics-v1",\n    "date": "2026-10-21",\n    "dataset_seeds": list(DATASET_SEEDS),\n    "optimizer_seeds": list(OPTIMIZER_SEEDS),\n    "protected_offsets": PROTECTED_OFFSETS,\n    "train_examples": TRAIN_EXAMPLES,\n    "development_examples": DEVELOPMENT_EXAMPLES,\n    "protected_examples": PROTECTED_EXAMPLES,\n    "history": HISTORY,\n    "horizon": HORIZON,\n    "state_dim": STATE_DIM,\n    "action_dim": ACTION_DIM,\n    "hidden_dim": HIDDEN_DIM,\n    "updates": UPDATES,\n    "batch_size": BATCH_SIZE,\n    "learning_rate": LEARNING_RATE,\n    "models": list(MODEL_NAMES),\n    "controls": ["direct", "residual", "persistence"],\n    "candidate": "uncertainty_residual",\n    "bootstrap_samples": BOOTSTRAP_SAMPLES,\n    "bootstrap_seed": BOOTSTRAP_SEED,\n    "max_wall_seconds": MAX_WALL_SECONDS,\n    "primary": {\n        "worst_regime_h8_ratio_max": WORST_REGIME_H8_RATIO_MAX,\n        "paired_delta_ci_upper_max": PAIRED_DELTA_CI_UPPER_MAX,\n        "clean_h8_regression_max": CLEAN_H8_REGRESSION_MAX,\n        "coverage90_error_max": COVERAGE90_ERROR_MAX,\n    },\n    "prior_protected_evidence_reused": False,\n    "paid_compute": False,\n    "promotion_authorized": False,\n}\n\n\ndef canonical(value: object) -> bytes:\n    return json.dumps(value, sort_keys=True, separators=(",", ":")).encode()\n\n\ndef digest(value: object) -> str:\n    return hashlib.sha256(canonical(value)).hexdigest()\n\n\ndef file_sha(path: Path) -> str:\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef write_new(path: Path, value: object) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("x", encoding="utf-8") as handle:\n        json.dump(value, handle, indent=2, sort_keys=True)\n        handle.write("\\n")\n\n\ndef resource_preflight() -> dict[str, Any]:\n    import psutil  # type: ignore[import-untyped]\n\n    memory = psutil.virtual_memory()\n    swap = psutil.swap_memory()\n    if memory.available < 2 * 1024**3:\n        raise RuntimeError("resource refusal: less than 2 GiB available")\n    if swap.total and swap.percent > 90.0:\n        raise RuntimeError("resource refusal: swap exceeds 90 percent")\n    return {\n        "available_bytes": memory.available,\n        "total_bytes": memory.total,\n        "swap_total_bytes": swap.total,\n        "swap_used_bytes": swap.used,\n        "swap_percent": swap.percent,\n    }\n\n\ndef source_identity() -> dict[str, str]:\n    source = Path(__file__).resolve()\n    protocol = source.parents[1] / "research/multi_horizon_dynamics_v1/PROTOCOL.md"\n    return {\n        "source_sha256": file_sha(source),\n        "protocol_sha256": file_sha(protocol),\n        "protocol_object_sha256": digest(PROTOCOL),\n    }\n\n\ndef main(output: Path) -> None:\n    started = time.monotonic()\n    output.mkdir(parents=True, exist_ok=False)\n    identities = source_identity()\n    preflight = resource_preflight()\n    freeze = {\n        "status": "FROZEN_BEFORE_PROTECTED_GENERATION",\n        "protocol": PROTOCOL,\n        "identities": identities,\n        "resource_preflight": preflight,\n        "protected_generated": False,\n    }\n    write_new(output / "SCIENTIFIC_FREEZE.json", freeze)\n\n    import torch\n    from torch import Tensor, nn\n\n    torch.set_num_threads(1)\n    torch.set_num_interop_threads(1)\n    torch.use_deterministic_algorithms(True)\n    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")\n\n    class Dynamics(nn.Module):\n        def __init__(self, mode: str) -> None:\n            super().__init__()\n            self.mode = mode\n            width = HIDDEN_DIM\n            self.encoder = nn.GRU(STATE_DIM + ACTION_DIM, width, batch_first=True)\n            self.cell = nn.GRUCell(STATE_DIM + ACTION_DIM, width)\n            self.mean = nn.Linear(width, STATE_DIM)\n            self.log_variance = (\n                nn.Linear(width, STATE_DIM)\n                if mode == "uncertainty_residual"\n                else None\n            )\n\n        def forward(self, history: Tensor, actions: Tensor) -> tuple[Tensor, Tensor | None]:\n            _, hidden = self.encoder(history)\n            cursor = hidden[-1]\n            state = history[:, -1, :STATE_DIM]\n            means: list[Tensor] = []\n            variances: list[Tensor] = []\n            for step in range(HORIZON):\n                cursor = self.cell(torch.cat((state, actions[:, step]), -1), cursor)\n                predicted = self.mean(cursor)\n                state = state + predicted if self.mode != "direct" else predicted\n                means.append(state)\n                if self.log_variance is not None:\n                    variances.append(self.log_variance(cursor).clamp(-8.0, 5.0))\n            return torch.stack(means, 1), torch.stack(variances, 1) if variances else None\n\n    def generate(seed: int, count: int, regime: str) -> dict[str, Tensor]:\n        generator = torch.Generator().manual_seed(seed)\n        actions = torch.randn(\n            count, HISTORY + HORIZON, ACTION_DIM, generator=generator\n        ) * 0.7\n        state = torch.randn(count, STATE_DIM, generator=generator)\n        states = [state]\n        coupling = torch.tensor(\n            [[0.0, 0.25, -0.10, 0.05], [-0.2, 0.0, 0.15, 0.1],\n             [0.12, -0.08, 0.0, 0.2], [0.05, 0.1, -0.18, 0.0]]\n        )\n        action_map = torch.tensor([[0.24, -0.05, 0.12, 0.08], [-0.08, 0.2, 0.04, -0.14]])\n        damping = 0.76 if regime != "shifted" else 0.64\n        nonlinear = 0.16 if regime != "shifted" else 0.24\n        for step in range(HISTORY + HORIZON - 1):\n            noise = torch.randn(count, STATE_DIM, generator=generator) * 0.015\n            drift = 0.025 * torch.sin(state * (1.0 + step / 10.0))\n            state = (\n                damping * state\n                + nonlinear * torch.tanh(state @ coupling)\n                + actions[:, step] @ action_map\n                + drift\n                + noise\n            )\n            states.append(state)\n        full = torch.stack(states, 1)\n        observed = full[:, :HISTORY].clone()\n        if regime == "corrupted":\n            observed[:, -2:] += torch.randn(\n                observed[:, -2:].shape, generator=generator\n            ) * 0.25\n            observed[:, -1, 1] = 0.0\n        history_actions = actions[:, :HISTORY]\n        return {\n            "history": torch.cat((observed, history_actions), -1),\n            "future_actions": actions[:, HISTORY - 1 : HISTORY - 1 + HORIZON],\n            "targets": full[:, HISTORY : HISTORY + HORIZON],\n            "last_clean": full[:, HISTORY - 1],\n        }\n\n    def train_model(\n        mode: str, dataset_seed: int, optimizer_seed: int\n    ) -> tuple[nn.Module, dict[str, Any]]:\n        torch.manual_seed(optimizer_seed)\n        random.seed(optimizer_seed)\n        model = Dynamics(mode)\n        optimizer = torch.optim.AdamW(\n            model.parameters(), lr=LEARNING_RATE, weight_decay=0.01\n        )\n        train = generate(dataset_seed + 1, TRAIN_EXAMPLES, "clean")\n        development = generate(dataset_seed + 2, DEVELOPMENT_EXAMPLES, "clean")\n        losses: list[float] = []\n        fit_started = time.monotonic()\n        for update in range(UPDATES):\n            index_generator = torch.Generator().manual_seed(optimizer_seed * 1000 + update)\n            indices = torch.randperm(TRAIN_EXAMPLES, generator=index_generator)[:BATCH_SIZE]\n            mean, log_variance = model(\n                train["history"][indices], train["future_actions"][indices]\n            )\n            error = mean - train["targets"][indices]\n            if log_variance is None:\n                loss = error.square().mean()\n            else:\n                loss = 0.5 * (error.square() * (-log_variance).exp() + log_variance).mean()\n            if not bool(torch.isfinite(loss)):\n                raise RuntimeError("non-finite training loss")\n            optimizer.zero_grad(set_to_none=True)\n            loss.backward()\n            nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            optimizer.step()\n            losses.append(float(loss.item()))\n        with torch.no_grad():\n            prediction, _ = model(development["history"], development["future_actions"])\n            development_mse = float((prediction - development["targets"]).square().mean().item())\n        return model, {\n            "mode": mode,\n            "dataset_seed": dataset_seed,\n            "optimizer_seed": optimizer_seed,\n            "updates": UPDATES,\n            "initial_loss": losses[0],\n            "final_loss": losses[-1],\n            "development_mse": development_mse,\n            "parameter_count": sum(parameter.numel() for parameter in model.parameters()),\n            "fit_seconds": time.monotonic() - fit_started,\n        }\n\n    checkpoints: dict[tuple[int, int, str], Path] = {}\n    fit_reports: list[dict[str, Any]] = []\n    for dataset_seed in DATASET_SEEDS:\n        for optimizer_seed in OPTIMIZER_SEEDS:\n            for mode in MODEL_NAMES:\n                if time.monotonic() - started > MAX_WALL_SECONDS:\n                    raise TimeoutError("campaign exceeded frozen wall budget")\n                model, report = train_model(mode, dataset_seed, optimizer_seed)\n                checkpoint = output / "checkpoints" / f"{dataset_seed}-{optimizer_seed}-{mode}.pt"\n                checkpoint.parent.mkdir(parents=True, exist_ok=True)\n                torch.save(model.state_dict(), checkpoint)\n                report["checkpoint_sha256"] = file_sha(checkpoint)\n                checkpoints[(dataset_seed, optimizer_seed, mode)] = checkpoint\n                fit_reports.append(report)\n    write_new(output / "FIT_REPORTS.json", fit_reports)\n\n    freeze["protected_generated"] = False\n    freeze["checkpoint_manifest_sha256"] = digest(\n        {str(path.relative_to(output)): file_sha(path) for path in checkpoints.values()}\n    )\n    write_new(output / "PROTECTED_ADMISSION.json", freeze)\n\n    rows: list[dict[str, Any]] = []\n    for dataset_seed in DATASET_SEEDS:\n        protected = {\n            regime: generate(\n                dataset_seed + PROTECTED_OFFSETS[regime],\n                PROTECTED_EXAMPLES,\n                regime,\n            )\n            for regime in PROTECTED_OFFSETS\n        }\n        for optimizer_seed in OPTIMIZER_SEEDS:\n            for mode in MODEL_NAMES:\n                model = Dynamics(mode)\n                state = torch.load(\n                    checkpoints[(dataset_seed, optimizer_seed, mode)],\n                    map_location="cpu",\n                    weights_only=True,\n                )\n                model.load_state_dict(state)\n                model.eval()\n                for regime, data in protected.items():\n                    with torch.no_grad():\n                        prediction, log_variance = model(\n                            data["history"], data["future_actions"]\n                        )\n                    squared = (prediction - data["targets"]).square()\n                    horizon_mse = squared.mean((0, 2)).tolist()\n                    row: dict[str, Any] = {\n                        "dataset_seed": dataset_seed,\n                        "optimizer_seed": optimizer_seed,\n                        "mode": mode,\n                        "regime": regime,\n                        "horizon_mse": horizon_mse,\n                        "mean_mse": float(squared.mean().item()),\n                        "recovery_error": float(squared[:, -1].mean().item()),\n                        "prediction_sha256": hashlib.sha256(\n                            prediction.contiguous().numpy().tobytes()\n                        ).hexdigest(),\n                    }\n                    if log_variance is not None:\n                        standard = (0.5 * log_variance).exp()\n                        inside = (prediction - data["targets"]).abs() <= 1.644854 * standard\n                        row["coverage90"] = float(inside.float().mean().item())\n                    rows.append(row)\n            for regime, data in protected.items():\n                prediction = data["last_clean"][:, None].expand(-1, HORIZON, -1)\n                squared = (prediction - data["targets"]).square()\n                rows.append(\n                    {\n                        "dataset_seed": dataset_seed,\n                        "optimizer_seed": optimizer_seed,\n                        "mode": "persistence",\n                        "regime": regime,\n                        "horizon_mse": squared.mean((0, 2)).tolist(),\n                        "mean_mse": float(squared.mean().item()),\n                        "recovery_error": float(squared[:, -1].mean().item()),\n                        "prediction_sha256": hashlib.sha256(\n                            prediction.contiguous().numpy().tobytes()\n                        ).hexdigest(),\n                    }\n                )\n    write_new(output / "PROTECTED_ROWS.json", rows)\n\n    def clustered(mode: str, regime: str, key: str) -> list[float]:\n        values = []\n        for dataset_seed in DATASET_SEEDS:\n            matched = [\n                row for row in rows\n                if row["dataset_seed"] == dataset_seed\n                and row["mode"] == mode\n                and row["regime"] == regime\n            ]\n            values.append(sum(float(row[key]) for row in matched) / len(matched))\n        return values\n\n    regimes = list(PROTECTED_OFFSETS)\n    models = [*MODEL_NAMES, "persistence"]\n    means = {\n        mode: {\n            regime: {\n                "mean_mse": sum(clustered(mode, regime, "mean_mse")) / 3,\n                "horizon8_mse": sum(\n                    float(row["horizon_mse"][-1]) for row in rows\n                    if row["mode"] == mode and row["regime"] == regime\n                ) / 6,\n            }\n            for regime in regimes\n        }\n        for mode in models\n    }\n    candidate = "uncertainty_residual"\n    learned_controls = ["direct", "residual"]\n    worst_regime = max(regimes, key=lambda value: means[candidate][value]["horizon8_mse"])\n    best_control = min(\n        learned_controls,\n        key=lambda value: means[value][worst_regime]["horizon8_mse"],\n    )\n    candidate_clusters = clustered(candidate, worst_regime, "recovery_error")\n    control_clusters = clustered(best_control, worst_regime, "recovery_error")\n    deltas = [candidate_clusters[index] - control_clusters[index] for index in range(3)]\n    rng = random.Random(BOOTSTRAP_SEED)\n    samples = []\n    for _ in range(BOOTSTRAP_SAMPLES):\n        selected = [deltas[rng.randrange(3)] for _ in range(3)]\n        samples.append(sum(selected) / 3)\n    samples.sort()\n    interval = [samples[250], samples[9749]]\n    coverage = sum(\n        float(row["coverage90"]) for row in rows\n        if row["mode"] == candidate and row["regime"] == worst_regime\n    ) / 6\n    candidate_h8 = means[candidate][worst_regime]["horizon8_mse"]\n    control_h8 = means[best_control][worst_regime]["horizon8_mse"]\n    clean_regression = (\n        means[candidate]["clean"]["horizon8_mse"]\n        - means["residual"]["clean"]["horizon8_mse"]\n    )\n    criteria = {\n        "worst_regime_ratio": candidate_h8 / control_h8,\n        "paired_delta_ci95": interval,\n        "clean_h8_regression": clean_regression,\n        "coverage90": coverage,\n    }\n    primary_supported = (\n        candidate_h8 / control_h8 <= WORST_REGIME_H8_RATIO_MAX\n        and interval[1] < PAIRED_DELTA_CI_UPPER_MAX\n        and clean_regression <= CLEAN_H8_REGRESSION_MAX\n        and abs(coverage - 0.9) <= COVERAGE90_ERROR_MAX\n    )\n    summary = {\n        "status": "COMPLETED_SUPPORTED" if primary_supported else "COMPLETED_NEGATIVE",\n        "primary_supported": primary_supported,\n        "worst_regime": worst_regime,\n        "best_learned_control": best_control,\n        "criteria": criteria,\n        "means": means,\n        "fits": len(fit_reports),\n        "protected_rows": len(rows),\n        "protocol_sha256": identities["protocol_sha256"],\n        "source_sha256": identities["source_sha256"],\n        "environment": {\n            "python": sys.version,\n            "platform": platform.platform(),\n            "torch": torch.__version__,\n        },\n        "elapsed_seconds": time.monotonic() - started,\n        "promotion_authorized": False,\n        "claim_boundary": "fresh bounded synthetic dynamics only",\n    }\n    write_new(output / "SUMMARY.json", summary)\n    write_new(\n        output / "MANIFEST.json",\n        {\n            str(path.relative_to(output)): file_sha(path)\n            for path in sorted(output.rglob("*"))\n            if path.is_file()\n        },\n    )\n    print(json.dumps(summary, indent=2, sort_keys=True))\n\n\nif __name__ == "__main__":\n    target = Path(\n        os.environ.get(\n            "MULTI_HORIZON_OUTPUT", "/kaggle/working/multi_horizon_dynamics_v1"\n        )\n    )\n    try:\n        main(target)\n    except BaseException as error:\n        target.mkdir(parents=True, exist_ok=True)\n        failure = target / f"FAILURE-{time.time_ns()}.json"\n        if not failure.exists():\n            write_new(\n                failure,\n                {\n                    "status": "FAILED_RETAINED",\n                    "error": repr(error),\n                    "torch_imported": "torch" in sys.modules,\n                    "protocol_sha256": digest(PROTOCOL),\n                },\n            )\n        raise\n', encoding='utf-8')
protocol.write_text('# Multi-horizon dynamics v1 prospective protocol\n\nStatus: **FROZEN BEFORE PROTECTED GENERATION**. Authorized by the user on\n2026-10-21 for one free Kaggle execution. This is a fresh synthetic successor;\nit does not reuse v5 checkpoints, examples, seeds, or protected outcomes.\n\n## Question and hypothesis\n\nCan a recurrent uncertainty-aware residual dynamics model improve long-horizon\nprediction under clean, corrupted-observation, and shifted-dynamics regimes\nrelative to identically structured direct and residual controls and a\npersistence baseline?\n\nThe primary hypothesis is supported only if the uncertainty-aware residual\ncandidate has at least 10% lower horizon-8 MSE than the best learned control in\nthe worst protected regime, the paired dataset-cluster bootstrap interval for\nthat difference is entirely below zero, clean horizon-8 MSE is no more than\n0.02 above the ordinary residual control, and empirical 90% interval coverage\nis within 0.10 of nominal. Failure of any criterion rejects the primary.\n\n## Data and separation\n\nThree dataset clusters use base seeds 20267100, 20267200, and 20267300. Each\ncluster has 512 training and 128 development trajectories. Protected clean,\ncorrupted-observation, and shifted-dynamics sets contain 256 trajectories each\nand use disjoint offsets +11, +12, and +13. Protected tensors are generated only\nafter the protocol/source freeze and all learned checkpoints are written.\n\nTrajectories have four state dimensions, two action dimensions, six observed\nhistory steps, and eight prediction horizons. The nonlinear controlled system,\nnoise levels, corruption, and shift are fixed in the source before execution.\n\n## Models and budget\n\nControls are persistence, recurrent direct prediction, and recurrent residual\nprediction. The candidate uses the residual architecture with a Gaussian\nnegative-log-likelihood objective and an explicit log-variance head. Learned\nmodels use two paired optimizer seeds, 200 AdamW updates, batch size 64, learning\nrate 0.002, one Torch thread, deterministic algorithms, and CPU execution.\nNo model or threshold selection uses protected outcomes.\n\nThe campaign is limited to 18 learned fits and 54 protected learned-model\nevaluations plus matched persistence evaluations. Wall time is capped at 3,600\nseconds. It must run on a free Kaggle CPU allocation with Internet disabled.\n\n## Reporting and validity\n\nReport every horizon-wise MSE, candidate interval coverage, recovery error,\nparameter count, elapsed fit time, checkpoint hash, prediction hash, and paired\ncluster delta. Use 10,000 percentile bootstrap samples with seed 20267999.\nSource/protocol/config identities, checkpoint files, raw cell reports, summary,\nenvironment, and failure receipts must be retained.\n\nAny source mismatch, overlapping seed, non-finite value, missing denominator,\nresource refusal, timeout, or post-freeze change makes the campaign INVALID.\nNegative results are retained. This study supports only a bounded synthetic\ndynamics conclusion; it cannot establish target-scale capability, SOTA,\ndeployment readiness, safety, or model promotion.\n', encoding='utf-8')
os.environ['MULTI_HORIZON_OUTPUT'] = '/kaggle/working/multi_horizon_dynamics_v1'
runpy.run_path(str(script), run_name='__main__')
